# ¿Qué puede hacer nuestra IA por nosotros?

En dos horas vamos a construir, celda a celda, un asistente sobre un proyecto pequeño: el
**software de gestión de trazas (TMS)** de un sistema de vigilancia costera. Tiene 16
requisitos, un documento de diseño en inglés, un manual del operador en español y unas 400
líneas de Ada. Es inventado (está en `datos/`), así que se puede mandar a cualquier modelo.

Todas las llamadas van al modelo de la empresa. Cada vez que algo falle lo apuntaremos en la
**lista de límites**; al final la ordenaremos.

| | Bloque | Minutos |
|---|---|---|
| 0 | Gancho | 0–5 |
| 1 | Qué es un modelo de lenguaje | 5–22 |
| 2 | Nuestros modelos | 22–32 |
| 3 | Que sepa lo nuestro | 32–55 |
| | *Descanso* | 55–65 |
| 4 | Preguntar con fuentes | 65–78 |
| 5 | Lo que no es semántico | 78–88 |
| 6 | Agentes | 88–103 |
| 7 | Límites y hacia dónde ir | 103–120 |

*Para ejecutarlo, el `README.md`. Las respuestas del modelo se guardan en `cache/`: se puede
volver a ejecutar sin acceso al modelo y sale lo mismo.*

## 0 · Gancho
*min 0–5 · Las dos caras del modelo.*

In [1]:
import json, math, re, sqlite3
import numpy as np
from curso import llm, data
from curso.show import table, bars, scatter, side_by_side, graph, tool, step, diagram, Limits

limits = Limits()                      # la lista de límites: crece durante la sesión
print(llm.describe())

profile openrouter: https://openrouter.ai/api/v1
  chat google/gemma-4-26b-a4b-it, embeddings baai/bge-m3, key sk-***
  to compare: google/gemma-4-26b-a4b-it, openai/gpt-oss-120b, mistralai/devstral-2512
  proxy: direct
  TLS: system certificate store
  28 requests/min; on 409/429 waits up to 15 min; timeout 600 s; cache on in cache/


Le damos un fichero Ada del proyecto y le pedimos que explique un procedimiento.

In [2]:
code = data.ada_sources()["src/track_store.adb"]
print(llm.ask(f"Explica en 3 líneas qué hace Purge_Stale:\n\n{code}"))

`Purge_Stale` recorre la tabla de seguimiento para identificar los registros que están actualmente en uso.
Compara el tiempo transcurrido desde el último reporte con un tiempo límite establecido (`Stale_Timeout`).
Si un registro ha superado dicho límite, lo elimina automáticamente llamando al procedimiento `Delete`.


> 💬 **Pregunta al chat:** ¿y si le preguntamos algo que no está en el código? ¿Qué creéis que hará?

In [3]:
assistant = "Eres el asistente técnico del proyecto TMS. Responde de forma directa."
print(llm.ask("¿Cuánto tarda en borrarse una traza según el manual del operador? "
              "¿Qué prueba lo verifica?", system=assistant))
limits.add("Inventa con seguridad lo que no sabe", "cualquier modelo")

Según el manual del operador, una traza tarda **24 horas** en borrarse. La prueba que lo verifica es la **prueba de ciclo de vida de la traza (Trace Lifecycle Test)**.


+ lista de límites: Inventa con seguridad lo que no sabe (cualquier modelo)

El manual dice «como mucho 3 segundos», y esa prueba no existe. Explica muy bien lo que tiene
delante e inventa con la misma seguridad lo que no. En dos horas veremos por qué pasan las dos
cosas y cómo se arregla la segunda.

## 1 · Qué es un modelo de lenguaje
*min 5–22 · Por qué inventa: una función de texto a texto, sin memoria, que da probabilidades.*

### Es HTTP con JSON

Un modelo de lenguaje es una función de texto a texto detrás de una URL. Esto es todo lo que
se manda; se podría mandar igual desde Ada.

In [4]:
request = {"model": llm.settings().model,
           "messages": [{"role": "user", "content": "Responde solo: OK"}]}
print("POST", llm.settings().base_url + "/chat/completions")
print(json.dumps(request, indent=2, ensure_ascii=False))

POST https://openrouter.ai/api/v1/chat/completions
{
  "model": "google/gemma-4-26b-a4b-it",
  "messages": [
    {
      "role": "user",
      "content": "Responde solo: OK"
    }
  ]
}


Y esto es todo lo que vuelve:

In [5]:
r = llm.chat(**request)
print(llm.to_json(r))

{
  "id": "gen-1791951510-M4q5hKycHQxU9MeWfGBq",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "OK",
        "role": "assistant"
      },
      "native_finish_reason": "stop"
    }
  ],
  "created": 1791951510,
  "model": "google/gemma-4-26b-a4b-it",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 2,
    "prompt_tokens": 18,
    "total_tokens": 20,
    "completion_tokens_details": {
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "image_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cache_write_tokens": 0,
      "cached_tokens": 0,
      "video_tokens": 0
    },
    "cost": 1.196e-06,
    "is_byok": false,
    "cost_details": {
      "upstream_inference_cost": 1.196e-06,
      "upstream_inference_prompt_cost": 7.56e-07,
      "upstream_inference_completions_cost": 4.4e-07
    }
  },
  "provider": "Darkbloom"
}


La respuesta está en `choices[0].message.content`, y lo que ha costado, en `usage`, contado en
**tokens**.

### Tokens

El modelo no ve letras ni palabras: ve tokens, trozos de texto. Los cuenta el propio servidor.

> 💬 **Pregunta al chat:** ¿cuántos tokens creéis que tiene «SRS-0004»?

In [6]:
EMPTY = llm.chat(".", max_tokens=1).usage.prompt_tokens - 1

def tokens(text):
    return llm.chat(text, max_tokens=1).usage.prompt_tokens - EMPTY

texts = ["The software shall delete a track within 2 s.",
         "El software borrará una traza en menos de 2 s.",
         "Ada.Text_IO.Put_Line", "SRS-0004", "Purge_Stale"]
table([{"texto": t, "caracteres": len(t), "tokens": tokens(t)} for t in texts])

texto,caracteres,tokens
The software shall delete a track within 2 s.,45,11
El software borrará una traza en menos de 2 s.,46,14
Ada.Text_IO.Put_Line,20,9
SRS-0004,8,6
Purge_Stale,11,5


Con `logprobs` el servidor devuelve cada token de la respuesta. Le pedimos que repita un texto
y miramos los trozos:

In [7]:
r = llm.chat("Repite exactamente, sin nada más: SRS-0004 Purge_Stale",
             temperature=0, logprobs=True)
print(" | ".join(t.token for t in r.choices[0].logprobs.content))

SRS | - | 0 | 0 | 0 | 4 |  Pur | ge | _ | St | ale


Los números van de uno en uno, y el español gasta más tokens que el inglés para decir lo
mismo. Importa porque todo se mide en tokens: lo que cabe en una petición, lo que tarda y lo
que cuesta.

### El siguiente token

Lo único que hace el modelo es dar, para cada posible siguiente token, una probabilidad. Le
damos una frase a medias del proyecto, sin ningún contexto.

In [8]:
r = llm.chat("Complete with one number, nothing else: "
             "The software shall delete a track within",
             max_tokens=1, temperature=0, logprobs=True, top_logprobs=5)
first = r.choices[0].logprobs.content[0]
bars([(a.token, math.exp(a.logprob)) for a in first.top_logprobs])

«3»   72.2 %
«1»   15.4 %
«2»    3.6 %
«6»    3.3 %
«one»    0.8 %

El requisito SRS-0004 dice **2 s**. El modelo no lo sabe: no consulta nada, elige lo más
probable. Una respuesta larga es eso mismo, token a token.

### Temperatura

Para elegir cada token se sortea entre los más probables. La temperatura decide cuánto: con 0
sale casi siempre el más probable; cuanto más alta, más variedad.

In [9]:
prompt = "Inventa un nombre para un paquete Ada que gestiona trazas de radar. Solo el nombre."
for t in (0.0, 1.2):
    names = [llm.ask(prompt, temperature=t, seed=i).strip() for i in range(5)]
    print(f"temperatura {t}: {', '.join(names)}")

temperatura 0.0: RadarTrace, RadarTrace, RadarTrace, RadarTrace, RadarTrace
temperatura 1.2: RadTrack, RadarTrace, RadarTrace, RadarTrace_Pkg, RadarTrace


### Sin memoria

Cada petición empieza de cero.

In [10]:
brief = "Responde en una frase."
print(llm.ask("Me llamo Ana y trabajo en el TMS.", system=brief))
print(llm.ask("¿Cómo me llamo?", system=brief))

history = [{"role": "user", "content": "Me llamo Ana y trabajo en el TMS."},
           {"role": "assistant", "content": "Encantado, Ana."},
           {"role": "user", "content": "¿Cómo me llamo?"}]
print(llm.chat(history).choices[0].message.content)
limits.add("No recuerda nada entre peticiones", "cualquier modelo")

Mucho gusto, Ana, es un placer saludarte y conocer que trabajas en el TMS.
No tengo acceso a tu información personal, por lo que no sé cómo te llamas.
Te llamas Ana.


+ lista de límites: No recuerda nada entre peticiones (cualquier modelo)

El «chat» es una lista de mensajes que se manda entera cada vez. Lo que cabe en esa lista (el
historial, los documentos que le pongamos y la respuesta) es la **ventana de contexto**: unos
200.000 tokens en el gateway. La respuesta tiene además su propio tope, `max_tokens`; si lo
alcanza, se corta:

In [11]:
r = llm.chat("Explica qué es Ada en un párrafo.", max_tokens=20)
print(r.choices[0].message.content)
print("\nfinish_reason:", r.choices[0].finish_reason)

**Ada** es un lenguaje de programación de alto nivel, fuertemente tipado y estructurado

finish_reason: length


### Por qué inventa

Predice el texto más plausible; no consulta nada. Si se le pregunta por algo que no existe, lo
más plausible es una respuesta que lo parezca. `Track_Store.Purge_All` no existe:

In [12]:
print(llm.ask("Escribe la declaración en Ada de Track_Store.Purge_All del TMS.",
              system=assistant))
limits.add("Inventa APIs, y en Ada más: ha visto mucho menos Ada que Python", "nuestros modelos")

La declaración de la firma del procedimiento en el paquete `Track_Store` es la siguiente:

```ada
procedure Purge_All;
```


+ lista de límites: Inventa APIs, y en Ada más: ha visto mucho menos Ada que Python (nuestros modelos)

## 2 · Nuestros modelos
*min 22–32 · Qué tenemos y por qué gemma-4 por defecto.*

| | gemma-4 | gpt-oss-120b | devstral |
|---|---|---|---|
| Tamaño | 26B, ~4B activos | 117B, ~5B activos | según la versión |
| Razonamiento | opcional | siempre | dentro de la respuesta |
| Pensado para | uso general, contexto largo | razonar | agentes de código |

**Mezcla de expertos**: muchos parámetros, pero cada token usa solo una parte. Los totales
guardan conocimiento (cuánto Ada ha visto); los activos marcan lo que cuesta cada token.

### Razonamiento

Algunos modelos «piensan» antes de responder: escriben un razonamiento que no se enseña, pero
que se paga en tokens y en tiempo, y que cuenta en `max_tokens`.

In [13]:
q = ("Una traza se borra como mucho 2 s después de pasar 10 s sin informes. ¿Cuántos "
     "segundos pueden pasar, como mucho, desde su último informe? Solo el número.")
for effort in ("none", "low"):
    r = llm.chat(q, reasoning_effort=effort)
    thinking = r.usage.completion_tokens_details.reasoning_tokens
    print(f"{effort:4}  respuesta {r.choices[0].message.content.strip():4}  "
          f"{thinking:4} tokens razonando  {llm.last['seconds']} s")

none  respuesta 12       0 tokens razonando  0.41 s
low   respuesta 12     545 tokens razonando  6.98 s


La misma respuesta, con cientos de tokens y varios segundos más. Razonar ayuda en problemas de
varios pasos, como escribir código; para esto, sobra.

### La misma tarea a los tres

`alarms.adb` tiene un fallo sembrado.

> 💬 **Pregunta al chat:** ¿cuál de los tres creéis que lo encuentra?

In [14]:
review = "¿Ves algún fallo en este paquete? Como mucho 5 líneas.\n\n"
alarms = data.ada_sources()["src/alarms.adb"]
side_by_side({model: (llm.ask(review + alarms, model=model), llm.last["seconds"])
              for model in llm.settings().models})

== google/gemma-4-26b-a4b-it (6.5 s)
El principal fallo es un **error de lógica de estado**:

Si `Now_Inside` es `True` y `Inside` ya es `True`, la alarma no se dispara (correcto), pero al final siempre haces `Inside (T.Id) := Now_Inside`. Si el track **sale** de la zona (`Now_Inside` es `False`), `Inside` se actualiza a `False`, pero **no hay una notificación de salida**. Si el requisito SRS-0007/0009 exige detectar cambios de estado (entrada y salida), te falta el `else if`.

== openai/gpt-oss-120b (3.0 s)
1. No se genera alarma ni registro al **salir** de una zona restringida; el requerimiento SRS‑0009 suele incluir notificación de salida.  
2. Si `Track_Id` no es un tipo discreto (por ejemplo, un tipo privado), el acceso `Inside (T.Id)` provocará una **Constraint_Error**.  
3. `Track_Id'Image` devuelve un espacio inicial; el mensaje del logger quedará con un espacio extra (“track 5”).  
4. No se capturan posibles excepciones de `Zones.In_Restricted_Zone`, lo que puede abortar la comprobación.  
5. El paquete no protege la variable global `Inside` contra accesos concurrentes en entornos multitarea.

== mistralai/devstral-2512 (1.4 s)
El principal fallo es que **no se inicializa el array `Inside` para todos los valores posibles de `Track_Id`** (solo se inicializa con `False` al declararlo, pero si `Track_Id` no cubre todos los índices, podría haber accesos fuera de rango).

Además, **falta manejar el caso cuando `Now_Inside` es `False` y `Inside(T.Id)` era `True`** (salida de zona restringida), lo que podría requerir notificar la salida (dependiendo de los requisitos SRS-0007/SRS-0009).

*Nota: Si `Track_Id` es un tipo enumerado completo, el primer punto no aplica.*

Ninguno. Cada uno encuentra *algo*, y todo suena razonable, pero el fallo no se ve en el
paquete: en este sistema los identificadores de traza **se reutilizan**, y eso lo dice el
diseño, no el código. Se lo damos: la sección 3 del diseño, delante del código.

In [15]:
architecture = data.documents()["tms_sdd.md"].split("## 3 Architecture")[1].split("## 4")[0]
side_by_side({model: (llm.ask(review + architecture + alarms, model=model), llm.last["seconds"])
              for model in llm.settings().models})
limits.add("Sin el contexto del sistema no ve el fallo, y señala otros que no lo son",
           "cualquier modelo")

== google/gemma-4-26b-a4b-it (1.9 s)
El fallo principal es una **condición de carrera o inconsistencia de estado**: si un `Track_Id` se reutiliza inmediatamente tras borrar un track, el valor previo en el array `Inside` podría hacer que no se detecte la entrada en la zona o que se dispare una alarma errónea. 

Además, el paquete `Alarms` no tiene un mecanismo para resetear su estado `Inside` cuando un track es eliminado en `Track_Store`.

== openai/gpt-oss-120b (3.2 s)
El paquete tiene un **error de estado**: el vector `Inside` nunca se limpia cuando una pista se elimina y su identificador vuelve a reutilizarse.  
Al reutilizar el mismo `Track_Id` el valor anterior (`True`) queda guardado, de modo que un nuevo objetivo que llegue ya fuera de zona no generará la alarma de “entrada” aunque la primera posición del reporte sí esté dentro, o viceversa.  

**Corrección:** en la rutina que borra una pista (p. ej. en `Track_Store`) debe ponerse `Inside (Id) := False;` o, alternativamente, `Alarms` debe exponer una operación `Reset (Id)` que el `Track_Store` invoque al liberar la entrada.  

Con eso el vector vuelve a reflejar el estado real de cada pista y el algoritmo de detección funciona correctamente.

== mistralai/devstral-2512 (1.2 s)
El fallo principal en el paquete `Alarms` es que **no inicializa el array `Inside` al crear nuevos tracks**. Dado que los `Track_Id` se reutilizan (según la descripción del sistema), un nuevo track podría heredar el valor `True` de un track anterior borrado, generando falsas alarmas.

**Solución rápida**: Añadir un procedimiento `Reset (T : Track_Id)` que ponga `Inside (T) := False;` y llamarlo desde `Track_Store` al crear un track.

+ lista de límites: Sin el contexto del sistema no ve el fallo, y señala otros que no lo son (cualquier modelo)

Con el contexto, los tres. Entre modelos cambian el estilo y el tiempo; lo que decide es lo que
tienen delante. De eso va el resto de la sesión.

## 3 · Que sepa lo nuestro
*min 32–55 · Cómo se le da al modelo el conocimiento que no tiene.*

### Pegarlo todo

La pregunta del día: **¿Cuánto tarda en borrarse una traza?** Primero, lo más simple: todo el
proyecto dentro del mensaje.

In [16]:
QUESTION = "¿Cuánto tarda en borrarse una traza?"
everything = ("\n\n".join(data.documents().values()) + "\n\n"
              + "\n".join(f"{r['id']}: {r['text']}" for r in data.requirements()))
print(llm.ask(f"{everything}\n\nPregunta: {QUESTION}", system=brief))
everything_tokens = llm.last["usage"]["prompt_tokens"]
print(f"\n{everything_tokens} tokens de entrada, {llm.last['seconds']} s")

El tiempo de borrado depende de la causa: si es por petición del operador tarda menos de 2 segundos, mientras que si es por pérdida de detección (traza obsoleta), tarda un máximo de 13 segundos tras dejar de recibirse el reporte (10 de espera más el tiempo de procesamiento).

3509 tokens de entrada, 1.21 s


Cabe, y contesta. Pero mira la respuesta: da 13 s (los 10 de espera más los 3 del manual) y no
dice que el requisito y el diseño hablan de 2 s. Con todo delante, no ve la contradicción.

Y cada pregunta manda el proyecto entero. La cuenta para un proyecto de verdad:

In [17]:
pages = len(everything) / 3000                    # unos 3000 caracteres por página
per_page = everything_tokens / pages
print(f"{per_page:.0f} tokens por página")
print(f"200 documentos de 50 páginas: {200 * 50 * per_page / 1e6:.0f} millones de tokens")

832 tokens por página
200 documentos de 50 páginas: 8 millones de tokens


Nuestro proyecto cabe; uno de verdad, no. Y aunque cupiera, cada pregunta tardaría y gastaría
en proporción. Hay que elegir los trozos que importan.

### Trocear

Primero, partir los documentos. Lo más simple es por tamaño fijo:

In [18]:
sdd = data.documents()["tms_sdd.md"]
fixed = [sdd[i:i + 1200] for i in range(0, len(sdd), 1200)]
k = next(i for i, c in enumerate(fixed) if "The worst-case time" in c)
print(f"{len(fixed)} trozos. El {k} acaba así:\n…{fixed[k][-170:]}")
print(f"\nY el {k + 1} empieza así:\n{fixed[k + 1][:120]}…")

7 trozos. El 3 acaba así:
…tor deletion does: `TRACK_DELETED` message to the HMI and record
in the event log.

The worst-case time from the moment a track becomes stale to its removal from the disp

Y el 4 empieza así:
lay
is therefore one purge period plus the HMI update time, below 1.1 s. This is within the 2 s
required by SRS-0004. Is…


Un trozo dice de qué se habla y el siguiente da la cifra: ninguno sirve solo. Solapar los
trozos lo suaviza; cortar por la estructura lo evita. Los documentos tienen secciones (aquí,
los `##` del Markdown), y el código, subprogramas: se corta por ahí.

In [19]:
def sections(name, text):
    parts = re.split(r"\n(?=##+ )", text)
    return [{"doc": name, "title": p.splitlines()[0].lstrip("# "), "text": p} for p in parts]

chunks = [c for name, text in data.documents().items() for c in sections(name, text)]
chunks += [{"doc": "requisitos", "title": r["id"], "text": f"{r['id']}: {r['text']}"}
           for r in data.requirements()]
print(len(chunks), "trozos")

41 trozos


### Buscar por palabras (BM25)

Lo de toda la vida. SQLite, que viene con Python, ya trae un índice de texto completo.

In [20]:
db = sqlite3.connect(":memory:")
db.execute("CREATE VIRTUAL TABLE fts USING fts5(text, tokenize='unicode61 remove_diacritics 2')")
db.executemany("INSERT INTO fts(rowid, text) VALUES (?, ?)",
               [(i, c["text"]) for i, c in enumerate(chunks)])

def bm25(query, k=5):
    words = " OR ".join(re.findall(r"\w+", query))
    rows = db.execute("SELECT rowid, -bm25(fts) FROM fts WHERE fts MATCH ? "
                      "ORDER BY bm25(fts) LIMIT ?", (words, k))
    return list(rows)

In [21]:
table([{"documento": chunks[i]["doc"], "sección": chunks[i]["title"], "puntos": round(s, 2)}
       for i, s in bm25(QUESTION)])

documento,sección,puntos
manual_operador.md,3 Las trazas,8.89
manual_operador.md,5 Alarmas,8.32
manual_operador.md,4 Borrar una traza,8.07
manual_operador.md,7 Preguntas frecuentes,5.8
manual_operador.md,6 Registro de eventos,4.03


Encuentra el manual, que comparte palabras con la pregunta («traza», «tarda»), aunque sea para
hablar de alarmas. El diseño no sale: está en inglés y habla de *stale tracks* y *purge cycle*.

### Buscar por significado (embeddings)

Un modelo de embeddings convierte cada texto en un vector. Textos que dicen lo mismo quedan
cerca, aunque usen otras palabras u otro idioma. La cercanía se mide con el coseno del ángulo
entre los dos vectores: tres líneas.

In [22]:
def cosine(a, b):
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

es, en, alarm = llm.embed(["Se borra la traza", "The track is deleted", "Se dispara una alarma"])
print(f"{len(es)} números por texto: {np.round(es[:4], 3)} …")
print(f"«Se borra la traza» y «The track is deleted»:   {cosine(es, en):.2f}")
print(f"«Se borra la traza» y «Se dispara una alarma»:  {cosine(es, alarm):.2f}")

1024 números por texto: [-0.005 -0.003 -0.064 -0.043] …
«Se borra la traza» y «The track is deleted»:   0.79
«Se borra la traza» y «Se dispara una alarma»:  0.53


La misma frase en otro idioma queda más cerca que otra frase en el mismo idioma. Lo mismo, con
todos los trozos y la pregunta:

In [23]:
vectors = llm.embed([c["text"] for c in chunks])
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

def semantic(query, k=5):
    q = llm.embed(query)[0]
    scores = vectors @ (q / np.linalg.norm(q))
    return [(i, float(scores[i])) for i in np.argsort(-scores)[:k]]

table([{"documento": chunks[i]["doc"], "sección": chunks[i]["title"], "coseno": round(s, 3)}
       for i, s in semantic(QUESTION)])

documento,sección,coseno
manual_operador.md,4 Borrar una traza,0.673
manual_operador.md,3 Las trazas,0.627
tms_sdd.md,4.4 Purge of stale tracks,0.6
manual_operador.md,7 Preguntas frecuentes,0.596
manual_operador.md,5 Alarmas,0.555


Ahora sí aparece el diseño. Los vectores tienen cientos de dimensiones; para verlos, se
proyectan en dos.

> 💬 **Pregunta al chat:** en el mapa, ¿junto a qué caerá la pregunta?

In [24]:
q = llm.embed(QUESTION)[0]
X = np.vstack([vectors, q / np.linalg.norm(q)])
X -= X.mean(axis=0)
xy = X @ np.linalg.svd(X, full_matrices=False)[2][:2].T        # PCA a 2 dimensiones
scatter(xy, labels=[c["title"] for c in chunks] + [QUESTION],
        groups=[c["doc"] for c in chunks] + ["pregunta"],
        highlight=[i for i, _ in semantic(QUESTION)] + [len(chunks)])

(-0.44, -0.32)  4 Borrar una traza  [manual_operador.md]
(-0.43, -0.15)  3 Las trazas  [manual_operador.md]
(-0.14, -0.23)  4.4 Purge of stale tracks  [tms_sdd.md]
(-0.42, -0.14)  7 Preguntas frecuentes  [manual_operador.md]
(-0.41, +0.05)  5 Alarmas  [manual_operador.md]
(-0.38, -0.47)  ¿Cuánto tarda en borrarse una traza?  [pregunta]

Con nombre, la pregunta y sus cinco trozos más cercanos. Al pasar a dos dimensiones el mapa
deforma las distancias, pero se ven tres familias: el manual, el diseño y los requisitos. La
pregunta cae junto al manual, que habla de lo mismo en su idioma, y el trozo del diseño más
cercano es justo el de la purga.

### Los identificadores

¿Y si lo que se busca es un identificador?

In [25]:
def ids(query):
    wanted = re.findall(r"\b[A-Z]+-\d+\b", query)
    return [i for i, c in enumerate(chunks)
            if any(re.search(rf"\b{w}\b", c["text"]) for w in wanted)]

table([{"por significado": chunks[i]["title"], "coseno": round(s, 3)}
       for i, s in semantic("SRS-0004", k=3)])
print("búsqueda exacta:", [f'{chunks[i]["doc"]} › {chunks[i]["title"]}' for i in ids("SRS-0004")])

por significado,coseno
SRS-0004,0.528
SRS-0005,0.528
SRS-0011,0.52


búsqueda exacta: ['tms_sdd.md › 4.4 Purge of stale tracks', 'tms_sdd.md › 8 Performance and capacity', 'requisitos › SRS-0004']


Por significado, un identificador se parece a todos los demás: acierta por los pelos, o no
acierta. La búsqueda exacta no falla nunca.

### Búsqueda híbrida

Cada técnica falla en algo. Se combinan las tres listas con *Reciprocal Rank Fusion*: cuenta
la posición de cada trozo en cada lista, no los puntos, que no se pueden comparar.

In [26]:
def fuse(rankings, k=5):
    score = {}
    for ranking in rankings:
        for pos, i in enumerate(ranking):
            score[i] = score.get(i, 0) + 1 / (60 + pos)
    return sorted(score, key=score.get, reverse=True)[:k]

def hybrid(query, k=5):
    return fuse([ids(query), [i for i, _ in semantic(query, 20)], [i for i, _ in bm25(query, 20)]], k)

table([{"documento": chunks[i]["doc"], "sección": chunks[i]["title"]} for i in hybrid(QUESTION)])
limits.add("Cada forma de buscar falla en algo", "cualquier modelo")

documento,sección
manual_operador.md,3 Las trazas
manual_operador.md,4 Borrar una traza
manual_operador.md,5 Alarmas
manual_operador.md,7 Preguntas frecuentes
manual_operador.md,1 Introducción


+ lista de límites: Cada forma de buscar falla en algo (cualquier modelo)

Para esta pregunta, el diseño vuelve a quedarse fuera: el manual sale arriba en dos de las tres
listas. Lo arreglamos en el bloque siguiente.

---

## Descanso
*min 55–65*

---

## 4 · Preguntar con fuentes
*min 65–78 · Responder solo con lo encontrado, citando.*

### Responder solo con lo encontrado

Buscar, poner los trozos numerados en el mensaje y pedirle que cite cada dato. Es el patrón
que más se usa (se llama RAG, *retrieval-augmented generation*).

In [27]:
RULES = ("Responde solo con las fuentes y cita cada dato como [n]. Si las fuentes no lo "
         "dicen, di que no lo sabes. Si dos fuentes dan cifras distintas para lo mismo, "
         "señálalo.")

def numbered(found):
    return "\n\n".join(f"[{n}] {chunks[i]['doc']} › {chunks[i]['title']}\n{chunks[i]['text']}"
                       for n, i in enumerate(found, 1))

def cited(found):
    print(*(f"[{n}] {chunks[i]['doc']} › {chunks[i]['title']}" for n, i in enumerate(found, 1)),
          sep="\n")

> 💬 **Pregunta al chat:** el manual dice 3 s y el requisito, 2 s. ¿Qué hará el modelo con dos cifras?

In [28]:
def answer(question, queries=None, k=6):
    found = fuse([hybrid(q, 10) for q in queries or [question]], k)
    print(llm.ask(f"Fuentes:\n{numbered(found)}\n\nPregunta: {question}", system=RULES))
    print(f"\n({llm.last['usage']['prompt_tokens']} tokens de entrada, {llm.last['seconds']} s)")
    cited(found)

answer(QUESTION)
limits.add("Si la búsqueda no trae el trozo, la respuesta falla", "cualquier modelo")

El tiempo que tarda en borrarse una traza depende de la situación:

*   **Borrado manual:** La traza desaparece de la pantalla en menos de 2 segundos [2].
*   **Borrado automático (cuando deja de detectarse):**
    *   Si el blanco deja de ser detectado, la traza permanece inmóvil durante 10 segundos [1].
    *   Una vez cumplidos esos 10 segundos, la traza tarda como mucho 3 segundos en desaparecer de la pantalla [1].
    *   Sin embargo, el documento técnico indica que el tiempo máximo desde que una traza se vuelve obsoleta hasta su eliminación de la pantalla es de menos de 1.1 segundos (un periodo de purga de 1 s más el tiempo de actualización de la HMI) [6]. El documento técnico también menciona que la versión 2 de dicho documento utilizaba un periodo de purga de 2 s [6].

(1045 tokens de entrada, 3.67 s)
[1] manual_operador.md › 3 Las trazas
[2] manual_operador.md › 4 Borrar una traza
[3] manual_operador.md › 5 Alarmas
[4] manual_operador.md › 7 Preguntas frecuentes
[5] manual_ope

+ lista de límites: Si la búsqueda no trae el trozo, la respuesta falla (cualquier modelo)

Cita bien, y con una fracción de los tokens de pegarlo todo. Pero casi todo lo que ha
encontrado es del manual: el diseño entra el último y el requisito, que dice 2 s, no llega. La
pregunta está en español, y el diseño y los requisitos, en inglés.

### Reescribir la pregunta

Antes de buscar, se le pide al modelo que convierta la pregunta en varias búsquedas, en los dos
idiomas. Con un glosario mínimo del proyecto: sin él, tradujo «traza» por *trace*, la de un
log, y no por *track*, la del radar.

In [29]:
GLOSSARY = "traza = track, informe = report, borrar = delete, zona restringida = restricted zone"

def rewrite(question):
    text = llm.ask("Escribe 3 consultas de búsqueda, en inglés y en español, para encontrar "
                   f"en la documentación técnica la respuesta a: {question}\n"
                   f"Glosario del proyecto: {GLOSSARY}. Una por línea, sin numerar ni comillas.")
    return [question] + [q.strip("-•*\"' ") for q in text.splitlines() if q.strip()]

queries = rewrite(QUESTION)
print("\n".join(queries), end="\n\n")
answer(QUESTION, queries)

¿Cuánto tarda en borrarse una traza?
How long does it take to delete a track? / ¿Cuánto tarda en borrarse una traza?
Track deletion time / Tiempo de borrado de una traza
Time required to delete a track / Tiempo requerido para borrar una traza

El tiempo de borrado de una traza depende de la causa del borrado:

*   **Borrado por parte del operador:** La traza desaparece de la pantalla en menos de 2 segundos [1]. El proceso completo (secuencia de la HMI al TMS) tarda menos de 100 ms [5].
*   **Borrado automático (por pérdida de detección):** Cuando el sensor deja de detectar un blanco, la traza permanece inmóvil durante 10 segundos [2]. Una vez cumplidos esos 10 segundos, la traza tarda como mucho 3 segundos en desaparecer de la pantalla [2]. 
*   **Cálculo de tiempo máximo (en el sistema):** El tiempo máximo desde que una traza se considera "obsoleta" (stale) hasta su eliminación de la pantalla es de un periodo de purga (1 s) más el tiempo de actualización de la HMI, lo cual es inferior

Ahora encuentra el diseño y avisa de que el manual (3 s) y el diseño (1,1 s) no dicen lo mismo.
Del requisito, en cambio, ha traído SRS-0006 y no SRS-0004: los dos dicen «delete a track
within 2 s», uno a petición del operador y otro por falta de informes. Por significado, casi no
se distinguen.

> 💬 **Pregunta al chat:** y si lo que preguntamos no está en ningún documento, ¿qué hará?

In [30]:
answer("¿Qué cifrado usan los mensajes al HMI?")

No lo sé.

(563 tokens de entrada, 1.65 s)
[1] requisitos › SRS-0042
[2] requisitos › SRS-0041
[3] tms_sdd.md › 7 HMI interface
[4] tms_sdd.md › 4.3 Deletion on operator request
[5] requisitos › SRS-0040
[6] requisitos › SRS-0010


«No lo sé» también es una respuesta, y con las fuentes delante la da. Sin ellas, se lo habría
inventado, como en el gancho.

### Verificar un requisito

SRS-0012 pide recuperar las trazas de los últimos 60 s tras un reinicio. ¿Qué dicen el diseño
y el manual? El requisito está en inglés y el manual en español: se busca con la pregunta
reescrita, y con más fuentes, porque hay que mirar en los dos documentos.

In [31]:
def verify(req_id, k=8):
    req = next(r for r in data.requirements() if r["id"] == req_id)
    found = fuse([hybrid(q, 10) for q in rewrite(f"{req_id}: {req['text']}")], k)
    print(llm.ask(f"Fuentes:\n{numbered(found)}\n\nRequisito {req_id}: {req['text']}\n"
                  "¿Lo cubren el diseño y el manual? Empieza por SÍ, NO o NO SE SABE, y "
                  "explícalo en 3 líneas.", system=RULES))
    cited(found)

verify("SRS-0012")

NO. El diseño indica que tras un reinicio la tabla de trazas queda vacía y se recrea conforme llegan reportes [2], mientras que el manual señala que la pantalla aparece vacía y se completa en pocos segundos [7], contradiciendo el requisito de restaurar los últimos 60 s [1].
[1] requisitos › SRS-0012
[2] tms_sdd.md › 9 Open issues
[3] requisitos › SRS-0044
[4] tms_sdd.md › 8 Performance and capacity
[5] requisitos › SRS-0008
[6] requisitos › SRS-0006
[7] manual_operador.md › 7 Preguntas frecuentes
[8] manual_operador.md › 3 Las trazas


El diseño lo deja abierto y el manual dice que tras un reinicio la pantalla aparece vacía: no se
cumple, y lo dice citando. Hacerlo con cada requisito y cada edición de los documentos es justo
el trabajo que nadie tiene tiempo de hacer a mano.

## 5 · Lo que no es semántico
*min 78–88 · Trazabilidad sin depender de que el modelo acierte.*

### ¿Qué código implementa SRS-0004?

Esto no es una pregunta para un modelo. El código ya lo dice: `--  Req: SRS-0004` encima de
cada subprograma que lo implementa y, en `tests/`, encima de cada prueba que lo verifica. Basta
una expresión regular.

In [32]:
def subprograms():
    for path, text in data.ada_sources().items():
        if not path.endswith(".adb"):
            continue
        unit = re.search(r"(?:package body|procedure) (\w+)", text).group(1)
        starts = list(re.finditer(r"^(?:   )?(?:procedure|function) (\w+)", text, re.M))
        for m, end in zip(starts, [s.start() for s in starts[1:]] + [len(text)]):
            before = text[:m.start()].rstrip().splitlines()[-1]
            name = unit if m.group(1) == unit else f"{unit}.{m.group(1)}"
            yield {"name": name, "body": text[m.start():end], "test": path.startswith("tests/"),
                   "reqs": re.findall(r"SRS-\d+", before) if "Req:" in before else []}

In [33]:
subs = list(subprograms())

def trace(req):
    return {"implementa": [s["name"] for s in subs if req in s["reqs"] and not s["test"]],
            "verifica": [s["name"] for s in subs if req in s["reqs"] and s["test"]]}

trace("SRS-0004")

{'implementa': ['Track_Store.Purge_Stale'],
 'verifica': ['Test_Track_Store.Test_Stale_Track_Is_Purged']}

Exacto, al instante y sin modelo. Y ya enseña algo: la prueba de SRS-0004 prueba `Purge_Stale`
por separado. Volveremos a ello en el bloque 6.

### Si cambio `Logger.Log`, ¿qué toco?

Las llamadas entre subprogramas forman un grafo. Subiendo por él, de llamador en llamador,
salen el código, las pruebas y los requisitos afectados.

In [34]:
names = {s["name"] for s in subs}
for s in subs:                         # Unit.Name (...) desde fuera, o Name (...) dentro de su unidad
    unit = s["name"].split(".")[0]
    s["calls"] = {n for n in names - {s["name"]}
                  if re.search(rf"\b{re.escape(n)}\s*\(", s["body"])
                  or (n.startswith(unit + ".") and
                      re.search(rf"(?<![.\w]){n.split('.')[-1]}\s*\(", s["body"]))}

def impact(name):
    seen, todo = {name}, [name]
    while todo:
        n = todo.pop()
        new = {s["name"] for s in subs if n in s["calls"]} - seen
        seen |= new
        todo += new
    return [s for s in subs if s["name"] in seen]

In [35]:
affected = impact("Logger.Log")
for label, test in (("código", False), ("pruebas", True)):
    print(f"{label}:", ", ".join(s["name"] for s in affected if s["test"] == test))
print("requisitos:", ", ".join(sorted({r for s in affected if not s["test"] for r in s["reqs"]})))
graph(affected, "Logger.Log")

código: Alarms.Check, Logger.Log, Tms_Main, Track_Store.Create, Track_Store.Delete, Track_Store.Purge_Stale
pruebas: Test_Alarms.Test_Alarm_On_Entry, Test_Alarms.Test_Alarm_Kept_Until_Acknowledged, Test_Track_Store.Test_Stale_Track_Is_Purged, Test_Track_Store.Test_Operator_Delete
requisitos: SRS-0003, SRS-0004, SRS-0006, SRS-0007, SRS-0008, SRS-0009, SRS-0011


Alarms.Check -> Logger.Log
Tms_Main -> Alarms.Check
Tms_Main -> Logger.Log
Tms_Main -> Track_Store.Create
Tms_Main -> Track_Store.Purge_Stale
Track_Store.Create -> Logger.Log
Track_Store.Delete -> Logger.Log
Track_Store.Purge_Stale -> Track_Store.Delete
Test_Alarms.Test_Alarm_On_Entry -> Alarms.Check
Test_Alarms.Test_Alarm_Kept_Until_Acknowledged -> Alarms.Check
Test_Track_Store.Test_Stale_Track_Is_Purged -> Track_Store.Create
Test_Track_Store.Test_Stale_Track_Is_Purged -> Track_Store.Purge_Stale
Test_Track_Store.Test_Operator_Delete -> Track_Store.Create
Test_Track_Store.Test_Operator_Delete -> Track_Store.Delete

### Huecos

Requisitos sin código, o con código y sin prueba:

In [36]:
done = {r for s in subs if not s["test"] for r in s["reqs"]}
tested = {r for s in subs if s["test"] for r in s["reqs"]}
table([{"requisito": r["id"], "código": "sí" if r["id"] in done else "—",
        "prueba": "sí" if r["id"] in tested else "—", "verificación": r["verification"]}
       for r in data.requirements() if r["id"] not in done or r["id"] not in tested])

requisito,código,prueba,verificación
SRS-0001,sí,—,Test
SRS-0002,—,—,Test
SRS-0003,sí,—,Test
SRS-0005,sí,—,Test
SRS-0008,sí,—,Test
SRS-0009,sí,—,Test
SRS-0011,sí,—,Analysis
SRS-0012,—,—,Test
SRS-0040,sí,—,Test
SRS-0041,sí,—,Test


Sin modelo, sin errores y al instante. **Determinista primero**; el modelo, para lo que sabe
hacer: leer, resumir, redactar. Por ejemplo, sacar las cifras de una sección del manual en un
formato fijo, con un esquema JSON que el servidor obliga a cumplir:

In [37]:
schema = {"type": "object", "additionalProperties": False, "required": ["cifras"],
          "properties": {"cifras": {"type": "array", "items": {
              "type": "object", "additionalProperties": False, "required": ["dato", "valor"],
              "properties": {"dato": {"type": "string"}, "valor": {"type": "string"}}}}}}
section = data.documents()["manual_operador.md"].split("## 3")[1].split("## 6")[0]
r = llm.chat(f"Saca las cifras que debe cumplir el sistema:\n{section}",
             response_format={"type": "json_schema",
                              "json_schema": {"name": "cifras", "schema": schema, "strict": True}})
print(json.dumps(json.loads(r.choices[0].message.content), indent=2, ensure_ascii=False))

{
  "cifras": [
    {
      "dato": "Tiempo de permanencia de la traza tras perder detección",
      "valor": "10 segundos"
    },
    {
      "dato": "Tiempo máximo de desaparición tras cumplirse el tiempo de permanencia",
      "valor": "3 segundos"
    },
    {
      "dato": "Tiempo máximo de desaparición tras pulsación de tecla SUPR",
      "valor": "2 segundos"
    },
    {
      "dato": "Tiempo máximo de aparición de alarma tras detección en zona restringida",
      "valor": "0,5 segundos"
    }
  ]
}


Sin esquema, pidiéndole JSON por favor:

In [38]:
text = llm.ask(f"Saca las cifras que debe cumplir el sistema, en JSON:\n{section}")
print(text[:400])
try:
    json.loads(text)
except json.JSONDecodeError as e:
    print("\njson.loads falla:", e)

Aquí tienes las cifras y tiempos especificados en el texto, estructurados en formato JSON:

```json
{
  "trazas": {
    "tiempo_permanencia_tras_perdida_deteccion_segundos": 10,
    "tiempo_maximo_desaparicion_tras_vencimiento_segundos": 3
  },
  "borrado_manual": {
    "tiempo_maximo_desaparicion_pantalla_segundos": 2
  },
  "alarmas": {
    "tiempo_maximo_aparicion_en_lista_segundos": 0.5
  }
}


json.loads falla: Expecting value: line 1 column 1 (char 0)


Para una persona, la respuesta es buena; para un programa, no sirve: viene envuelta en texto,
y los nombres de los campos se los inventa cada vez. Con el esquema, el servidor solo deja salir
JSON válido con esos campos.

## 6 · Agentes
*min 88–103 · Un agente es un bucle. Qué le pasa con Ada.*

### Darle herramientas

Un agente es un bucle: el modelo pide una herramienta, la ejecutamos, le damos el resultado, y
así hasta que responde. Las herramientas son lo que ya hemos construido.

In [39]:
TOOLS = {
    "search": lambda query: [f"{chunks[i]['doc']} › {chunks[i]['title']}:\n{chunks[i]['text']}"
                             for i in hybrid(query, 3)],
    "get_requirement": lambda id: next(r["text"] for r in data.requirements() if r["id"] == id),
    "read_file": lambda path: data.ada_sources().get(path, f"no existe; hay: {list(data.ada_sources())}"),
    "trace": lambda id: trace(id),
}
SCHEMAS = [tool("search", "Busca en requisitos y documentos", query="qué buscar"),
           tool("get_requirement", "Texto oficial de un requisito", id="p. ej. SRS-0004"),
           tool("read_file", "Lee un fichero Ada", path="p. ej. src/tms_main.adb"),
           tool("trace", "Código y pruebas de un requisito", id="p. ej. SRS-0004")]

El modelo no ve el código de las herramientas: ve su nombre, para qué sirven y sus parámetros,
en JSON.

In [40]:
print(json.dumps(SCHEMAS[2], indent=2, ensure_ascii=False))

{
  "type": "function",
  "function": {
    "name": "read_file",
    "description": "Lee un fichero Ada",
    "parameters": {
      "type": "object",
      "required": [
        "path"
      ],
      "properties": {
        "path": {
          "type": "string",
          "description": "p. ej. src/tms_main.adb"
        }
      }
    }
  }
}


In [41]:
def agent(task, max_steps=15, **params):
    messages = [{"role": "system", "content": "Revisas código Ada. Usa las herramientas; no supongas."},
                {"role": "user", "content": task}]
    for _ in range(max_steps):
        message = llm.chat(messages, tools=SCHEMAS, **params).choices[0].message
        messages.append(message)
        if not message.tool_calls:
            return message.content
        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            result = TOOLS[call.function.name](**args)
            step(call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return "(se acabaron los pasos)"

> 💬 **Pregunta al chat:** la prueba de SRS-0004 pasa. ¿Cumple el código el requisito?

In [42]:
print(agent("¿Cumple el código el requisito SRS-0004 en todos los casos?"))

→ get_requirement(id='SRS-0004')
    The software shall delete a track within 2 s when no report has been received for that track for 10 s.

→ trace(id='SRS-0004')
    {
     "implementa": [
      "Track_Store.Purge_Stale"
     ],
     "verifica": [
      "Test_Track_Store.Test_Stale_Track_Is_Purged"
     ]
    }

→ read_file(path='src/track_store.adb')
    with Ada.Text_IO;
    with HMI;
    with Logger;
    
    package body Track_Store is
    
       Table : array (Track_Id) of Track;
    
       --  Req: SRS-0003, SRS-0011
       procedure Create
         (Target : Positive;
          Pos    : Position;
          Now    : Seconds;
          Id     : out Track_Id;
          Ok     : out Boolean) is
       be

→ read_file(path='src/track_store.ads')
    with Tracks; use Tracks;
    
    --  The table of current tracks.
    package Track_Store is
    
       procedure Create
         (Target : Positive;
          Pos    : Position;
          Now    : Seconds;
          Id     : out Track_Id;
          Ok     : out Boolean);
       --  Creates a track for a new target. Ok is False if the table

→ search(query='Stale_Timeout Purge_Period')
    [
     "tms_sdd.md › 4.4 Purge of stale tracks:\n### 4.4 Purge of stale tracks\n\nA target that stops being detected leaves behind a track that no longer receives reports.\nKeeping such a track on the display for a short time is useful, because the target may be\ndetected again; keeping it too long misl


No, el código **no cumple** con el requisito SRS-0004 en todos los casos debido a una discrepancia en la lógica de comparación de tiempos en la implementación de `Purge_Stale`.

### Análisis del Incumplimiento

1.  **Requisito SRS-0004:**
    *"The software shall delete a track within 2 s when no report has been received for that track for 10 s."*
    (El software debe eliminar una pista en un plazo de 2 s cuando no se haya recibido ningún informe para esa pista durante 10 s).

2.  **Implementación en `track_store.adb`:**
    ```ada
    --  Req: SRS-0004
    procedure Purge_Stale (Now : Seconds) is
    begin
       for I in Table'Range loop
          if Table (I).In_Use
            and then Now - Table (I).Last_Report >= Stale_Timeout -- <--- ERROR LÓGICO
          then
             Delete (I);
          end if;
       end loop;
    end Purge_Stale;
    ```

3.  **El Problema:**
    La condición `Now - Table (I).Last_Report >= Stale_Timeout` es correcta para identificar una pista "sta

Se acercó: dice que cumplir los 2 s depende de cómo llame el bucle principal a la purga. Pero
no fue a mirarlo. Y empieza diciendo «no cumple» para acabar diciendo «es funcionalmente
correcto».

In [43]:
limits.add("Sin razonar, no comprueba lo que sospecha y se contradice", "nuestros modelos")

+ lista de límites: Sin razonar, no comprueba lo que sospecha y se contradice (nuestros modelos)

### Dejarle pensar

Lo mismo, con la configuración que funcionó escribiendo Ada: razonamiento `low` y temperatura
1,0.

> 💬 **Pregunta al chat:** ¿lo encontrará ahora?

In [44]:
print(agent("¿Cumple el código el requisito SRS-0004 en todos los casos?",
            reasoning_effort="low", temperature=1.0))
limits.add("Un agente gasta muchas peticiones por tarea, y el cupo es de 28 por minuto",
           "nuestra infraestructura")

→ get_requirement(id='SRS-0004')
    The software shall delete a track within 2 s when no report has been received for that track for 10 s.

→ trace(id='SRS-0004')
    {
     "implementa": [
      "Track_Store.Purge_Stale"
     ],
     "verifica": [
      "Test_Track_Store.Test_Stale_Track_Is_Purged"
     ]
    }

→ read_file(path='src/track_store.adb')
    with Ada.Text_IO;
    with HMI;
    with Logger;
    
    package body Track_Store is
    
       Table : array (Track_Id) of Track;
    
       --  Req: SRS-0003, SRS-0011
       procedure Create
         (Target : Positive;
          Pos    : Position;
          Now    : Seconds;
          Id     : out Track_Id;
          Ok     : out Boolean) is
       be

→ read_file(path='src/track_store.ads')
    with Tracks; use Tracks;
    
    --  The table of current tracks.
    package Track_Store is
    
       procedure Create
         (Target : Positive;
          Pos    : Position;
          Now    : Seconds;
          Id     : out Track_Id;
          Ok     : out Boolean);
       --  Creates a track for a new target. Ok is False if the table

→ read_file(path='tests/test_track_store.adb')
    with Ada.Text_IO; use Ada.Text_IO;
    with Tracks;      use Tracks;
    with Track_Store;
    
    --  Unit tests of Track_Store.
    procedure Test_Track_Store is
    
       Failures : Natural := 0;
    
       procedure Check (Condition : Boolean; Name : String) is
       begin
          if Condition then
             Put_Line ("PASS " & Name

...

→ read_file(path='src/tracks.ads')
    --  Track data shared by every package of the Track Management Software.
    package Tracks is
    
       Max_Tracks : constant := 1_000;
    
       type Track_Id is range 1 .. Max_Tracks;
    
       type Seconds is new Duration;
       --  Time since the software started.
    
       type Position is record
          X, Y : Float := 0.0;

→ read_file(path='src/sensor_input.ads')
    with Tracks; use Tracks;
    
    --  Reports from the primary sensor. This version reads them from standard
    --  input, one per line: target number, X and Y in metres.
    package Sensor_Input is
    
       type Report is record
          Target : Positive := 1;
          Pos    : Position;
       end record;
    
       procedure Read (

→ read_file(path='src/sensor_input.adb')
    with Ada.Text_IO;
    with Ada.Integer_Text_IO;
    with Ada.Float_Text_IO;
    
    package body Sensor_Input is
    
       --  Req: SRS-0001
       procedure Read (R : out Report; Available : out Boolean) is
          Target : Integer;
          X, Y   : Float;
          Last   : Natural;
       begin
          if Ada.Text_IO.End_Of_File then

No, el código **no cumple** el requisito **SRS-0004** en todos los casos.

### Análisis del incumplimiento

El requisito **SRS-0004** establece:
> *"El software deberá eliminar una pista en un plazo de 2 s cuando no se haya recibido ningún informe para esa pista durante 10 s."*

Para que esto se cumpla, el proceso de purga (`Purge_Stale`) debe ejecutarse periódicamente, independientemente de si llegan nuevos informes o no, garantizando que la limpieza ocurra en un intervalo máximo de 2 segundos tras cumplirse los 10 segundos de silencio.

Sin embargo, la implementación en `src/tms_main.adb` vincula la ejecución de la purga a la recepción de informes del sensor:

```ada
-- src/tms_main.adb
loop
   Sensor_Input.Read (Report, Available); -- Bloqueante
   exit when not Available;
   declare
      Now : constant Seconds := Clocks.Now;
   begin
      -- ... actualización de pistas ...
      if Now >= Next_Purge then
         Track_Store.Purge_Stale (Now); -- Solo se llama si Read() retorna
 

+ lista de límites: Un agente gasta muchas peticiones por tarea, y el cupo es de 28 por minuto (nuestra infraestructura)

Ahora sí. Lee el bucle principal y la entrada del sensor: `Tms_Main` se queda esperando en
`Sensor_Input.Read` y solo purga después de un informe, así que si el sensor se calla, ninguna
traza se borra. Y explica por qué la prueba no lo ve: prueba `Purge_Stale` por separado. **La
prueba pasa y el requisito no se cumple.**

Por el camino pidió `src/main.adb`, que no existe; la herramienta le contestó qué ficheros hay
y siguió. A cambio, once peticiones para una sola pregunta.

### Las mismas herramientas, en el editor

MCP (*Model Context Protocol*) es la forma estándar de ofrecer herramientas como estas a
cualquier editor o agente: VS Code, OpenCode y otros las descubren y las usan con el modelo que
tengan configurado. Se escriben una vez y sirven para todos.

In [45]:
diagram("mcp")

El editor VS Code, OpenCode… <-> Servidor MCP del proyecto search · get_requirement read_file · trace
Servidor MCP del proyecto search · get_requirement read_file · trace -> Requisitos, documentos y código Ada
El editor VS Code, OpenCode… <-> El modelo de la empresa

### Lo que pasó cuando escribió un servidor en Ada

Un servidor de comandos por sockets: 25 requisitos, 544 líneas, 10 pruebas. Con gemma-4,
encargo a encargo y con una persona al lado, salió entero y las pruebas pasan. Hizo falta
razonamiento `low` y temperatura 1,0, encargos de una sola cosa, ponerle en las instrucciones
lo que no sabía (la API de `GNAT.Sockets`, reglas de Ada en las que fallaba) y compilar fuera
del agente. **Sin ayuda humana, ninguna configuración llegó a un servidor que funcione.**
devstral escribe unas 8 veces más rápido, pero no depura errores de ejecución.

In [46]:
limits.add("Se inventó la API de GNAT.Sockets", "nuestros modelos")
limits.add("Dijo dos veces que compilaba sin ser así", "nuestros modelos")
limits.add("Con encargos amplios entra en bucles", "nuestros modelos")
limits.add("Al pedirle arreglar una prueba que falla, cambia la prueba", "nuestros modelos")
limits.add("No tiene acceso a nuestro código, requisitos ni documentos", "nuestra infraestructura")

+ lista de límites: Se inventó la API de GNAT.Sockets (nuestros modelos)

+ lista de límites: Dijo dos veces que compilaba sin ser así (nuestros modelos)

+ lista de límites: Con encargos amplios entra en bucles (nuestros modelos)

+ lista de límites: Al pedirle arreglar una prueba que falla, cambia la prueba (nuestros modelos)

+ lista de límites: No tiene acceso a nuestro código, requisitos ni documentos (nuestra infraestructura)

## 7 · Límites y hacia dónde ir
*min 103–120 · Qué usar ya, qué no y qué pedir.*

### La lista de límites

In [47]:
limits.show()

De cualquier modelo (5):
  - Inventa con seguridad lo que no sabe
  - No recuerda nada entre peticiones
  - Sin el contexto del sistema no ve el fallo, y señala otros que no lo son
  - Cada forma de buscar falla en algo
  - Si la búsqueda no trae el trozo, la respuesta falla
  Lo arregla: Software: fuentes, grafo, herramientas; compilar y probar fuera del modelo

De nuestros modelos (6):
  - Inventa APIs, y en Ada más: ha visto mucho menos Ada que Python
  - Sin razonar, no comprueba lo que sospecha y se contradice
  - Se inventó la API de GNAT.Sockets
  - Dijo dos veces que compilaba sin ser así
  - Con encargos amplios entra en bucles
  - Al pedirle arreglar una prueba que falla, cambia la prueba
  Lo arregla: Modelos más capaces para programar

De nuestra infraestructura (2):
  - Un agente gasta muchas peticiones por tarea, y el cupo es de 28 por minuto
  - No tiene acceso a nuestro código, requisitos ni documentos
  Lo arregla: Capacidad, acceso a las fuentes e integración en el editor

La primera columna es la más importante: **un modelo mejor no la arregla**. Hace falta montarlo
bien, y eso se puede hacer ya.

### Lo que dicen las medidas

- **Para conocimiento y trazabilidad, los modelos actuales ya valen.** 24 preguntas de
  respuesta conocida sobre un proyecto de ejemplo, con fuentes: 24 aciertos, en 5 o 6 s cada
  una. Lo que falta es conectarlos a lo nuestro.
- **Para programar con agentes, falta modelo.** Con encargos pequeños y una persona al lado,
  sale; sin ella, no.

### Hacia dónde ir

1. **Integración primero.** Conectar la IA a nuestros documentos, requisitos y código, y
   llevarla al editor. Es barato, se puede hacer ya y con los modelos actuales ya rinde.
2. **Capacidad.** Con 28 peticiones por minuto para todos, un equipo no puede usar agentes:
   hacen falta cupos por persona.
3. **Mejores modelos para programar.** Depende de una pregunta: **¿puede salir nuestro código
   de la empresa?** Si sí, con contrato, modelos punteros. Si no, modelos abiertos más grandes
   en hardware propio. En los dos casos, decidirlo con tareas nuestras en Ada, no con
   benchmarks de Python.

### El semáforo

- 🟢 **Útil ya:** explicar código ajeno; buscar y preguntar a la documentación, con citas;
  resumir y redactar.
- 🟡 **Con revisión:** escribir pruebas y código pequeño, un encargo cada vez; comprobar un
  requisito contra un documento; revisar un cambio.
- 🔴 **Nunca solo:** fiarse de «compila» o «pasa» sin haberlo ejecutado; dejarle arreglar la
  prueba que falla; encargos grandes sin supervisión.

### Lo que hemos construido

In [48]:
diagram("sesion")

Pregunta -> Reescribir con glosario
Reescribir con glosario -> Búsqueda híbrida (RRF)
Búsqueda híbrida (RRF) -> Fuentes numeradas
Fuentes numeradas -> Modelo
Modelo -> Respuesta con citas [n]
Requisitos, diseño, manual -> Trozos por sección
Trozos por sección -> IDs exactos, vectores, BM25
IDs exactos, vectores, BM25 -> Búsqueda híbrida (RRF)
Código Ada -> Grafo de llamadas y de «Req:»
Grafo de llamadas y de «Req:» -> Traza, impacto, huecos
Agente: un bucle -> Herramientas
Herramientas --> Traza, impacto, huecos
Herramientas --> IDs exactos, vectores, BM25
Agente: un bucle <-> Modelo
Herramientas --> El editor, por MCP

Cada caja ha salido de una celda de este notebook, con el modelo que ya tenemos. Lo que falta
para usarlo de verdad no es un modelo mejor: es montarlo sobre nuestros documentos, nuestros
requisitos y nuestro código, y llevarlo al editor.